# 🐼 LeetCode Pandas Practice

This notebook contains my solutions from the **30 Days of Pandas** track on LeetCode.

The goal is to practice data manipulation, transformation, and analysis using pandas.

---

## 📊 Data Filtering

### 595. Big Countries (Easy)

Find countries with a large area or population.

**Solution:**


In [ ]:
# Filter countries by area or population threshold
def big_countries(world: pd.DataFrame) -> pd.DataFrame:
    return world.loc[(world['area'] >= 3000000) | (world['population'] >=25000000),['name','population', 'area']]

---

### 1757. Recyclable and Low Fat Products (Easy)

Find products that are both low fat and recyclable.

**Solution:**

In [ ]:
# Filter only products that are low fat and recyclable only
def find_products(products: pd.DataFrame) -> pd.DataFrame:
    return products.loc[
        (products["low_fats"] == "Y") & (products["recyclable"] == "Y"), ["product_id"]
    ]

---

### 183. Customers Who Never Order (Easy)

Find customers who have never placed an order.

**Solution:**

In [ ]:
# Get all the customers that never placed and order
def find_customers(customers: pd.DataFrame, orders: pd.DataFrame) -> pd.DataFrame:
    return customers.loc[~customers["id"].isin(orders["customerId"]), ["name"]].rename(
        columns={"name": "Customers"}
    )

---

### 1148. Article Views I (Easy)

Find authors who viewed their own articles.

**Solution:**

In [ ]:
# Filter only authors that viewed their own articles
def article_views(views: pd.DataFrame) -> pd.DataFrame:
    return (
        views.loc[views["author_id"] == views["viewer_id"], ["author_id"]]
        .drop_duplicates()
        .rename(columns={"author_id": "id"})
        .sort_values(by="id")
    )

---

### 1149. Article Views II (Medium)

Find users who viewed more than one article on the same date.

**Solution:**

In [ ]:
# Group by viewer and data, then count unique number of article viewed by a user
def article_views(views: pd.DataFrame) -> pd.DataFrame:
    df = (
        views.groupby(["viewer_id", "view_date"])["article_id"]
        .nunique()
        .reset_index(name="count")
    )
    return (
        df.loc[df["count"] > 1, ["viewer_id"]]
        .drop_duplicates()
        .rename(columns={"viewer_id": "id"})
        .sort_values(by="id")
    )

---

## 🔤 String Methods

### 1683. Invalid Tweets (Easy)

Find tweets with more than 15 characters.

**Solution:**

In [ ]:
# Filter only the tweets that its content is greater than 15
def invalid_tweets(tweets: pd.DataFrame) -> pd.DataFrame:
    return tweets.loc[tweets["content"].str.len() > 15, ["tweet_id"]]

---

### 1873. Calculate Special Bonus (Easy)

Calculate bonus based on employee ID and name rules.

**Solution:**

In [ ]:
# Filter using the employee_id and the name of the employees to calculate the bonus
def calculate_special_bonus(employees: pd.DataFrame) -> pd.DataFrame:
    employees["bonus"] = 0
    employees.loc[
        (employees["employee_id"] % 2 != 0) & ~(employees["name"].str.startswith("M")),
        ["bonus"],
    ] = employees["salary"]
    return employees[["employee_id", "bonus"]].sort_values(by="employee_id")

---

### 1667. Fix Names in a Table (Easy)

Format names: first letter uppercase, rest lowercase.

**Solution:**

In [ ]:
# Format the names of the users
def fix_names(users: pd.DataFrame) -> pd.DataFrame:
    users["name"] = users["name"].str.capitalize()
    return users.sort_values(by="user_id")

---

### 1517. Find Users With Valid E-Mails (Easy)

Find users with valid emails that match the required pattern.

**Solution:**

In [ ]:
# Filter the valid mails using a regex pattern
def valid_emails(users: pd.DataFrame) -> pd.DataFrame:
    pattern = r"^[A-Za-z][A-Za-z0-9\-_\.]*@leetcode\.com$"
    return users.loc[users["mail"].str.fullmatch(pattern)]

---

### 1527. Patients With a Condition (Easy)

Find patients with Type I Diabetes (DIAB1).

**Solution:**

In [ ]:
# Filter coditions that contains DIAB1 code
def find_patients(patients: pd.DataFrame) -> pd.DataFrame:
    return patients[
        patients["conditions"].str.contains(r"(^|\s)DIAB1\d*", na=False, regex=True)
    ]

---

### 2738. Count Occurrences in Text (Medium)

Count files that contain the words "bull" and "bear" as standalone words.

**Solution:**

In [ ]:
# Count the occurrences of the word bull and bear, counting only the one that are standalone words, and not are in the beginning or end of the content
def count_occurrences(files: pd.DataFrame) -> pd.DataFrame:
    words = ["bull", "bear"]
    return pd.DataFrame(
        {
            "word": words,
            "count": [
                files["content"].str.contains(rf"(?<=\s){w}(?=\s)").sum() for w in words
            ],
        }
    )

---

## 🔄 Data Manipulation

### 177. Nth Highest Salary (Medium)

Find the nth highest distinct salary.

**Solution:**

In [ ]:
# Filter and find the nth highest distinct salary
def nth_highest_salary(employee: pd.DataFrame, N: int) -> pd.DataFrame:
    employee = employee["salary"].drop_duplicates().sort_values(ascending=False)
    return pd.DataFrame(
        {
            f"getNthHighestSalary({N})": [
                None if (N > employee.size) | (N <= 0) else employee.iloc[N - 1]
            ]
        }
    )

---

### 176. Second Highest Salary (Medium)

Find the second highest distinct salary.

**Solution:**

In [ ]:
# Find the second highest distinct salary, returning None if there's no a second highest salary
def second_highest_salary(employee: pd.DataFrame) -> pd.DataFrame:
    employee = employee["salary"].drop_duplicates().sort_values(ascending=False)
    return pd.DataFrame(
        {"SecondHighestSalary": [None if (employee.size < 2) else employee[1]]}
    )

---

### 184. Department Highest Salary (Medium)

Find employees with the highest salary in each department.

**Solution:**

In [ ]:
# Group by department and using transform to get the max salary in each department, and filter the employees with the highest salary
def department_highest_salary(
    employee: pd.DataFrame, department: pd.DataFrame
) -> pd.DataFrame:
    df = pd.merge(
        employee,
        department,
        how="left",
        left_on="departmentId",
        right_on="id",
        suffixes=["_emp", "_dept"],
    )
    max_salaries = df.groupby("departmentId")["salary"].transform("max")
    return df.loc[
        df["salary"] == max_salaries, ["name_dept", "name_emp", "salary"]
    ].rename(
        columns={"name_dept": "Department", "name_emp": "Employee", "salary": "Salary"}
    )